# M3 Module A: chuẩn bị 5 bảng đầu vào (Colab CPU)

**Cách chạy (ai trong nhóm cũng chạy được):** không cần sửa gì, notebook tự tìm thư mục `arXiv-data` trong Drive của bạn, tự chọn nhánh code và tự đặt tên run mới `m3-inputs-NN`. Cần: thư mục chung `arXiv-data` đã được chia sẻ với quyền chỉnh sửa và đã *Thêm lối tắt vào Drive*. Bấm **Runtime → Restart session**, sau đó **Runtime → Run all**. Cell nào lỗi thì dừng lại và báo lỗi rõ ràng; sửa xong thì chạy lại từ cell 1.

Notebook chỉ **đọc** dữ liệu M1/M2 trên Drive. Kết quả ghi vào `/content/<RUN_ID>` rồi chép lên Drive `milestone 3/inputs/<RUN_ID>` (không bao giờ ghi đè thư mục đã có).

In [ ]:
# 1. Cấu hình + mount Drive. Ai trong nhóm cũng chạy được: để trống thì notebook tự dò.
RUN_ID = ''         # để trống = tự lấy tên mới m3-inputs-NN chưa có trên Drive; hoặc ghi rõ, ví dụ 'm3-inputs-02'
BRANCH = ''         # để trống = dùng nhánh huynh-nhi/m3-data-prep nếu còn, không thì main (sau khi PR đã merge)
DRIVE_PATH = ''     # để trống = tự tìm thư mục arXiv-data trong Drive của bạn
REPO_URL = 'https://github.com/ChickenDev-cyber/arxiv-data-science'
SAMPLE_PAPERS = 3                    # số bài mỗi owner trong sample
XOA_BAN_CU_TREN_CONTENT = True       # True: xoá /content/<RUN_ID> cũ (chỉ bản tạm trên Colab, không đụng Drive)

import json, os, shutil, subprocess
from pathlib import Path

os.chdir('/content')
from google.colab import drive
drive.mount('/content/drive')


def tim_arxiv_data():
    """Tìm thư mục arXiv-data (có Full-data và milestone 2) trong Drive đã mount của người đang chạy."""
    goc = Path('/content/drive')
    ung_vien = [goc / 'MyDrive' / 'arXiv-data']
    ung_vien += sorted(goc.glob('.shortcut-targets-by-id/*/arXiv-data'))
    ung_vien += sorted(goc.glob('Shareddrives/*/arXiv-data'))
    ung_vien += sorted(goc.glob('MyDrive/*/arXiv-data')) + sorted(goc.glob('MyDrive/*/*/arXiv-data'))
    for p in ung_vien:
        if (p / 'Full-data').is_dir() and (p / 'milestone 2').is_dir():
            return p
    raise FileNotFoundError(
        'Không tìm thấy thư mục arXiv-data trong Drive. Mở thư mục chung trên drive.google.com, '
        'bấm Sắp xếp -> Thêm lối tắt vào Drive (Add shortcut to Drive) -> Drive của tôi, '
        'rồi chạy lại cell này. Hoặc điền DRIVE_PATH thủ công.')


DRIVE = Path(DRIVE_PATH) if DRIVE_PATH else tim_arxiv_data()
INPUTS = DRIVE / 'milestone 3' / 'inputs'
if not RUN_ID:
    n = 1
    while (INPUTS / f'm3-inputs-{n:02d}').exists():
        n += 1
    RUN_ID = f'm3-inputs-{n:02d}'
if not BRANCH:
    co_nhanh = subprocess.run(['git', 'ls-remote', '--exit-code', '--heads', REPO_URL, 'huynh-nhi/m3-data-prep'],
                              capture_output=True).returncode == 0
    BRANCH = 'huynh-nhi/m3-data-prep' if co_nhanh else 'main'
REPO = Path('/content/repo')
CONFIG = Path('/content/m3_sources.json')
M1_LOCAL = Path('/content/m1')
OUT = Path('/content') / RUN_ID
SAMPLE = Path('/content') / f'{RUN_ID}-sample'
SAMPLE_ZIP = Path('/content') / f'{RUN_ID}-sample.zip'
DEST = INPUTS / RUN_ID


def sh(cmd, cwd=REPO, ok=(0,)):
    """Chạy lệnh bash, in output ngay khi có, báo lỗi nếu exit code không nằm trong ok."""
    env = dict(os.environ, PYTHONPATH='src')
    proc = subprocess.Popen(['bash', '-c', cmd], cwd=str(cwd), env=env, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end='')
    code = proc.wait()
    print(f'exit={code}')
    if code not in ok:
        raise RuntimeError(f'Lệnh lỗi (exit {code}): {cmd}')
    return code


assert (DRIVE / 'Full-data').is_dir() and (DRIVE / 'milestone 2').is_dir(), f'Sai DRIVE_PATH: {DRIVE}'
assert not DEST.exists(), f'{DEST} đã có trên Drive. Đổi RUN_ID rồi chạy lại cell này.'
print('Drive:', DRIVE)
print('RUN_ID =', RUN_ID, '| nhánh code:', BRANCH)
print('output tạm:', OUT, '| lưu Drive:', DEST)

In [ ]:
# 2. Lấy code mới nhất của nhánh + cài thư viện + chạy test Module A
assert 'RUN_ID' in globals(), 'Chưa chạy cell 1 (cấu hình). Chạy cell 1 trước.'
os.chdir('/content')                 # đứng ngoài repo trước khi xoá
shutil.rmtree(REPO, ignore_errors=True)
sh(f'git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}', cwd='/content')
sh('pip -q install pyarrow==23.0.1 pandas numpy')
sh('git log -1 --oneline')
sh('python -m unittest tests.test_matching_data 2>&1 | tail -n 5')

In [ ]:
# 3. Tự dò Full-data/*.zip (M1) và milestone 2/<owner>/<run>/handoff.json (M2) -> /content/m3_sources.json
assert 'RUN_ID' in globals(), 'Chưa chạy cell 1 (cấu hình). Chạy cell 1 trước.'
assert REPO.is_dir(), 'Chưa có code. Chạy cell 2 trước.'
sh(f'python -m arxiv_pipeline.matching.data scan --drive-root "{DRIVE}" --write "{CONFIG}"')
cfg = json.loads(CONFIG.read_text())
thieu = [o for o, s in cfg['owners'].items() if not (s.get('m1_corpus') and (s.get('m2_run_dir') or s.get('m2_parquet')))]
print('Owner thiếu M1 hoặc M2:', thieu or 'không có')

**Kiểm tra output trên:** mỗi owner phải có `m1_corpus` và `m2_run_dir`. Nếu một owner có nhiều run M2, hỏi nhóm run nào là chính thức rồi sửa `/content/m3_sources.json` (thanh Files bên trái) trước khi chạy tiếp.

In [ ]:
# 4. Chép 5 zip M1 về ổ local (khoảng 3,7 GB) để đọc nhanh, rồi trỏ config sang bản local
assert 'RUN_ID' in globals(), 'Chưa chạy cell 1 (cấu hình). Chạy cell 1 trước.'
assert CONFIG.exists(), 'Chưa có /content/m3_sources.json. Chạy cell 3 (scan) trước.'
cfg = json.loads(CONFIG.read_text())
M1_LOCAL.mkdir(exist_ok=True)
for owner, spec in cfg['owners'].items():
    src = spec.get('m1_corpus')
    if src and src.startswith('/content/drive'):
        dst = M1_LOCAL / Path(src).name
        if not dst.exists() or dst.stat().st_size != Path(src).stat().st_size:
            print('copy', src)
            shutil.copy(src, str(dst) + '.part')     # chép dở thì không bị nhận nhầm là xong
            os.replace(str(dst) + '.part', dst)
        spec['m1_snapshot'] = Path(src).stem
        spec['m1_corpus'] = str(dst)
CONFIG.write_text(json.dumps(cfg, ensure_ascii=False, indent=2))
print(json.dumps(cfg['owners'], ensure_ascii=False, indent=2))

In [ ]:
# 5. Chạy chính (khoảng 20 phút). Cuối output phải có: module E contract (contracts.load_inventory): OK
assert 'RUN_ID' in globals(), 'Chưa chạy cell 1 (cấu hình). Chạy cell 1 trước.'
assert CONFIG.exists(), 'Chưa có /content/m3_sources.json. Chạy cell 3 và 4 trước.'
if OUT.exists():
    assert XOA_BAN_CU_TREN_CONTENT, f'{OUT} đã tồn tại. Đổi RUN_ID hoặc đặt XOA_BAN_CU_TREN_CONTENT = True'
    print('xoá bản cũ trên Colab:', OUT)
    shutil.rmtree(OUT)
code = sh(f'python -m arxiv_pipeline.matching.data prepare --config "{CONFIG}" --output-dir "{OUT}"', ok=(0, 2))
assert (OUT / 'inventory.json').exists(), f'Không thấy {OUT}/inventory.json'
if code == 2:
    raise RuntimeError('Có owner lỗi: xem các dòng [error] ở trên, chi tiết trong inventory.json. Chưa nên tạo sample/lưu Drive.')
print('OK:', OUT / 'inventory.json')

Exit `0` = cả 5 owner `ok` và inventory đạt contract module E. Exit `2` = có owner thiếu/lỗi (cell báo lỗi và dừng). Cảnh báo `[warning]` không chặn bàn giao nhưng cần ghi lại.

In [ ]:
# 6. Sample nhỏ từ dữ liệu thật (ưu tiên bài có cả bibtex và references) + nén zip
assert 'RUN_ID' in globals(), 'Chưa chạy cell 1 (cấu hình). Chạy cell 1 trước.'
assert (OUT / 'inventory.json').exists(), f'Chưa có {OUT}/inventory.json. Chạy cell 5 (prepare) trước.'
shutil.rmtree(SAMPLE, ignore_errors=True)
SAMPLE_ZIP.unlink(missing_ok=True)
sh(f'python -m arxiv_pipeline.matching.data sample --inventory "{OUT}/inventory.json" --output-dir "{SAMPLE}" --papers {SAMPLE_PAPERS}')
shutil.make_archive(str(SAMPLE), 'zip', root_dir='/content', base_dir=SAMPLE.name)
print(SAMPLE_ZIP, f'{SAMPLE_ZIP.stat().st_size / 1e6:.1f} MB')

In [ ]:
# 7. Lưu lên Drive (thư mục mới, không ghi đè), kiểm tra đủ file, rồi flush để Drive ghi xong thật
assert 'RUN_ID' in globals(), 'Chưa chạy cell 1 (cấu hình). Chạy cell 1 trước.'
assert (OUT / 'inventory.json').exists() and SAMPLE_ZIP.exists(), 'Chưa có output/sample. Chạy cell 5 và 6 trước.'
assert not DEST.exists(), f'{DEST} đã có trên Drive. Không ghi đè: đổi RUN_ID và chạy lại từ cell 1.'
shutil.copytree(OUT, DEST)
shutil.copy(SAMPLE_ZIP, DEST.parent / SAMPLE_ZIP.name)
shutil.copy(CONFIG, DEST / 'm3_sources.json')
n_local = sum(1 for p in OUT.rglob('*') if p.is_file())
n_drive = sum(1 for p in DEST.rglob('*') if p.is_file()) - 1   # trừ m3_sources.json
print('số file local / Drive:', n_local, n_drive, n_local == n_drive)
assert n_local == n_drive, 'Chép thiếu file. Xoá thư mục trên Drive rồi chạy lại cell này.'
print('saved to', DEST)
print('sample:', DEST.parent / SAMPLE_ZIP.name)
drive.flush_and_unmount()
print('Xong. Gửi link file sample ở trên cho nhóm.')